# Lab 06 — Explaining Models with LIME

LIME explains one prediction at a time. It changes the input a little many times, watches how the model's answer changes, and fits a simple linear model to those results.

- **Q1:** Breast Cancer data, Random Forest, `LimeTabularExplainer`
- **Q2:** IMDb reviews, TF-IDF + Logistic Regression, `LimeTextExplainer`
- **Q3:** a dog photo, MobileNetV2, `LimeImageExplainer`

Sections marked ★ are extra experiments.

In [ ]:
import os, re, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # silence TensorFlow C++ logs
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
POS, NEG = "#2e7d32", "#c62828"   # green = supports, red = opposes (used in every plot)

import sklearn, lime, tensorflow as tf
print("sklearn", sklearn.__version__, "| lime", getattr(lime, "__version__", "0.2.x"), "| tensorflow", tf.__version__)

---
# Question 1 — LIME for Tabular Data

## 1.1 Load the dataset
Load the Breast Cancer dataset from sklearn. It has 569 tumours with 30 numeric features. Label 0 is malignant and 1 is benign.

In [ ]:
from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
X = pd.DataFrame(bc.data, columns=bc.feature_names)
y = pd.Series(bc.target, name="target")
class_names = [str(c) for c in bc.target_names]          # ['malignant', 'benign']

print("Shape:", X.shape)
print("Class counts:", dict(zip(class_names, np.bincount(y))))
X.iloc[:, :10].describe().T.round(2)

## 1.2 Pre-processing and split
Check for missing values and duplicates. Scaling is skipped because Random Forests don't need it. Split 80/20 and keep the class ratio the same in both parts.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score

print("Missing values:", int(X.isna().sum().sum()), "| duplicate rows:", int(X.duplicated().sum()))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Malignant share  train %.3f | test %.3f" % ((y_train == 0).mean(), (y_test == 0).mean()))

## 1.3 Train a Random Forest
Train a default Random Forest, then compare its accuracy on the training data and the test data. A big gap means overfitting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_base = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_train, y_train)
print(f"Default RF  train acc {rf_base.score(X_train, y_train):.4f} | test acc {rf_base.score(X_test, y_test):.4f}")

### 1.3b Fix the overfitting
The default forest gets 100% on training data, so it is memorising. We try limits on tree depth and leaf size with 5-fold cross-validation on the training data only. Among settings within one standard error of the best score, we keep the one with the smallest train–validation gap.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def pick_simplest(search):
    # among settings within one standard error of the best CV score, keep the smallest train-validation gap
    r = pd.DataFrame(search.cv_results_)
    r["gap"] = r.mean_train_score - r.mean_test_score
    best = r.loc[r.mean_test_score.idxmax()]
    tol = best.std_test_score / np.sqrt(search.n_splits_)
    ok = r[r.mean_test_score >= best.mean_test_score - tol]
    chosen = ok.loc[ok.gap.idxmin()]
    assert chosen.mean_test_score >= best.mean_test_score - tol
    return r, chosen

rf_search = GridSearchCV(
    RandomForestClassifier(n_estimators=300, random_state=SEED),
    {"max_depth": [2, 3, 5, 8, None], "min_samples_leaf": [1, 3, 5, 10]},
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    return_train_score=True, refit=False, n_jobs=-1).fit(X_train, y_train)
rf_cv, rf_choice = pick_simplest(rf_search)
print("Chosen settings:", rf_choice.params)

# validation curve over depth, at the chosen leaf size
sub = rf_cv[rf_cv.param_min_samples_leaf == rf_choice.params["min_samples_leaf"]]
xl = ["full" if d is None else str(d) for d in sub.param_max_depth]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(xl, sub.mean_train_score, "o-", label="train")
ax.plot(xl, sub.mean_test_score, "s-", label="cross-validation")
ax.fill_between(xl, sub.mean_test_score - sub.std_test_score, sub.mean_test_score + sub.std_test_score, alpha=0.2)
ax.set_xlabel("max_depth"); ax.set_ylabel("accuracy"); ax.legend()
ax.set_title(f"RF fit check (min_samples_leaf={rf_choice.params['min_samples_leaf']})")
plt.tight_layout(); plt.show()

rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, **rf_choice.params).fit(X_train, y_train)
default_row = rf_cv[rf_cv.param_max_depth.isna() & (rf_cv.param_min_samples_leaf == 1)].iloc[0]
fit_tab = pd.DataFrame({
    "train acc": [rf_base.score(X_train, y_train), rf.score(X_train, y_train)],
    "CV acc": [default_row.mean_test_score, rf_choice.mean_test_score],
    "test acc": [rf_base.score(X_test, y_test), rf.score(X_test, y_test)]},
    index=["default RF", f"tuned RF {rf_choice.params}"])
fit_tab["train-test gap"] = fit_tab["train acc"] - fit_tab["test acc"]
display(fit_tab.round(4))

## 1.4 Evaluate the model
Report accuracy, precision, recall and the confusion matrix for the tuned model. Precision and recall use malignant as the positive class, because missing a cancer is the worst mistake.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

y_pred = rf.predict(X_test)
p_malig = rf.predict_proba(X_test)[:, 0]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (malignant)": precision_score(y_test, y_pred, pos_label=0),
    "Recall (malignant)": recall_score(y_test, y_pred, pos_label=0),
    "F1 (malignant)": f1_score(y_test, y_pred, pos_label=0),
    "ROC-AUC": roc_auc_score(y_test == 0, p_malig),
}
display(pd.Series(metrics, name="value").round(4).to_frame())
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4.6, 3.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=class_names, yticklabels=class_names, ax=ax, annot_kws={"size": 14})
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Random Forest — confusion matrix (test)")
plt.tight_layout(); plt.show()

## 1.5 Explain one prediction with LIME
LIME creates thousands of slightly changed copies of one patient's row and asks the model about each one. It then fits a small linear model whose weights show how much each feature mattered. We explain a malignant test case.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

feature_names = list(X.columns)
def rf_predict(a):
    # LIME passes numpy arrays; wrap them so the forest sees the column names it was trained with
    return rf.predict_proba(pd.DataFrame(a, columns=feature_names))

tab_explainer = LimeTabularExplainer(
    X_train.values, feature_names=feature_names, class_names=class_names,
    mode="classification", discretize_continuous=True, random_state=SEED)

idx = int(np.where((y_pred == 0) & (y_test.values == 0))[0][0])   # first correctly-predicted malignant case
x0 = X_test.iloc[idx].values
proba0 = rf_predict(x0[None])[0]
label0 = int(proba0.argmax())
print(f"Test row {idx} | actual: {class_names[y_test.iloc[idx]]} | predicted: {class_names[label0]}")
print("RF probabilities:", dict(zip(class_names, proba0.round(3))))

exp_tab = tab_explainer.explain_instance(x0, rf_predict, labels=(label0,), num_features=10, num_samples=5000)
print(f"LIME local model: intercept={exp_tab.intercept[label0]:.3f}, "
      f"local prediction={exp_tab.local_pred[0]:.3f}, R²={exp_tab.score:.3f}")

### Plot 1 — feature importance
How strongly each feature affected this prediction, ignoring direction.

In [ ]:
tab_list = exp_tab.as_list(label=label0)
imp = pd.DataFrame(tab_list, columns=["rule", "weight"])
imp["abs"] = imp.weight.abs()
imp = imp.sort_values("abs")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(imp.rule, imp["abs"], color="#1565c0")
ax.set_xlabel("|LIME weight|"); ax.set_title(f"LIME feature importance — test row {idx} ({class_names[label0]})")
plt.tight_layout(); plt.show()

### Plot 2 — positive and negative contributions
Green features push towards the predicted class. Red features push away from it.

In [ ]:
contrib = pd.DataFrame(tab_list, columns=["rule", "weight"]).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(contrib.rule, contrib.weight, color=[POS if w > 0 else NEG for w in contrib.weight])
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel(f"LIME weight  (→ {class_names[label0]}   ← {class_names[1 - label0]})")
ax.set_title("LIME explanation: supporting (green) vs opposing (red) features")
plt.tight_layout(); plt.show()

## 1.6 Top five features
The five features with the largest LIME weights, next to the patient's actual values.

In [ ]:
top5_idx = [f for f, w in exp_tab.as_map()[label0][:5]]
top5 = pd.DataFrame({
    "feature": [feature_names[f] for f in top5_idx],
    "rule": [r for r, w in tab_list[:5]],
    "instance value": [x0[f] for f in top5_idx],
    "train median": [X_train.iloc[:, f].median() for f in top5_idx],
    "LIME weight": [w for r, w in tab_list[:5]],
})
top5["effect"] = np.where(top5["LIME weight"] > 0, f"→ {class_names[label0]}", f"→ {class_names[1 - label0]}")
display(top5.round(4))

Large, irregular cell nuclei push the prediction towards malignant, which matches medical knowledge.

---
## ★ 1.7 Is the explanation stable?
LIME is random, so run it 10 times at different sample sizes and measure how much the top-5 features overlap (1 = identical).

In [ ]:
from itertools import combinations

def top_k_set(exp, k=5):
    return {f for f, w in exp.as_map()[label0][:k]}

rows = []
for n in [500, 1000, 5000]:
    sets, t = [], time.time()
    for s in range(10):
        e = LimeTabularExplainer(X_train.values, feature_names=feature_names, class_names=class_names,
                                 discretize_continuous=True, random_state=s
                                 ).explain_instance(x0, rf_predict, labels=(label0,), num_features=5, num_samples=n)
        sets.append(top_k_set(e))
    jacc = [len(a & b) / len(a | b) for a, b in combinations(sets, 2)]
    rows.append({"num_samples": n, "mean Jaccard top-5": np.mean(jacc), "min Jaccard": np.min(jacc),
                 "sec per explanation": (time.time() - t) / 10})
stab = pd.DataFrame(rows)
display(stab.round(3))
assert stab["mean Jaccard top-5"].between(0, 1).all()

More samples give more stable explanations. Features that are almost copies of each other (radius, perimeter, area) can swap places between runs.

## ★ 1.8 Local vs global importance
Compare three model-wide importance measures: the forest's built-in importance, permutation importance, and LIME averaged over 40 test rows.

In [ ]:
from sklearn.inspection import permutation_importance

imp_rf = pd.Series(rf.feature_importances_, index=feature_names)
perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=SEED, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=feature_names)

agg = pd.Series(0.0, index=feature_names)
sample_rows = np.random.RandomState(SEED).choice(len(X_test), 40, replace=False)
for r in sample_rows:
    e = tab_explainer.explain_instance(X_test.values[r], rf_predict, labels=(0,), num_features=30, num_samples=1000)
    for f, w in e.as_map()[0]:
        agg.iloc[f] += abs(w) / len(sample_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, (name, s) in zip(axes, [("RF impurity importance", imp_rf),
                                ("Permutation importance (test)", imp_perm),
                                ("Aggregated LIME |weight| (40 rows)", agg)]):
    top = s.sort_values().tail(10)
    ax.barh(top.index, top.values, color="#6a1b9a")
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

ranks = pd.DataFrame({"impurity": imp_rf, "permutation": imp_perm, "agg_LIME": agg}).rank(ascending=False)
print("Spearman rank correlation between the three importance measures:")
display(ranks.corr(method="spearman").round(3))

The built-in and averaged-LIME rankings mostly agree. Permutation importance differs because correlated features can stand in for each other.

## ★ 1.9 Deletion test
Replace the top LIME features with typical values and see how much the prediction drops, compared with replacing random features.

In [ ]:
order = [f for f, w in exp_tab.as_map()[label0]]           # LIME ranking (top-10)
medians = X_train.median().values
rng = np.random.RandomState(SEED)

def p_after_removing(feats):
    x = x0.copy(); x[list(feats)] = medians[list(feats)]
    return rf_predict(x[None])[0, label0]

ks = range(0, 11)
lime_curve = [p_after_removing(order[:k]) for k in ks]
rand_curve = [np.mean([p_after_removing(rng.choice(30, k, replace=False)) for _ in range(30)]) for k in ks]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(ks, lime_curve, "o-", color=NEG, label="remove LIME top-k features")
ax.plot(ks, rand_curve, "s--", color="gray", label="remove k random features (mean of 30)")
ax.set_xlabel("k features replaced by training median"); ax.set_ylabel(f"P({class_names[label0]})")
ax.set_title("Deletion test — faithfulness of the tabular explanation"); ax.legend()
plt.tight_layout(); plt.show()
print("P(malignant) after removing LIME top-5: %.3f vs random-5: %.3f" % (lime_curve[5], rand_curve[5]))

---
# Question 2 — LIME for Text Data

## 2.1 Load IMDb
Load 50,000 IMDb reviews from Keras. They are stored as word numbers, so decode them back into text.

In [ ]:
from tensorflow.keras.datasets import imdb

(xa, ya), (xb, yb) = imdb.load_data()
word_index = imdb.get_word_index()
inv_index = {i + 3: w for w, i in word_index.items()}

def decode(seq):
    return " ".join(inv_index.get(i, "") for i in seq[1:])   # seq[0] is the <start> token

reviews = pd.DataFrame({"review": [decode(s) for s in np.concatenate([xa, xb])],
                        "label": np.concatenate([ya, yb])})
print(reviews.shape, "| positive share: %.3f" % reviews.label.mean())
print(reviews.review[0][:400], "...")

## 2.2 Clean the text
Drop empty reviews and duplicates. Lower-case the text, remove leftover HTML and symbols, and squash extra spaces. Stop words are kept because words like "not" matter for sentiment.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<br\s*/?>|\bbr\b", " ", text)       # HTML line breaks
    text = re.sub(r"[^a-z' ]+", " ", text)              # unnecessary symbols / digits
    text = re.sub(r"\s+", " ", text).strip()
    return text

assert clean_text("Great<br /><br />film!! 10/10 br br don't") == "great film don't"

reviews["review"] = reviews.review.replace(r"^\s*$", np.nan, regex=True)
print("Missing reviews:", int(reviews.review.isna().sum()))
reviews = reviews.dropna(subset=["review"])
reviews["clean"] = reviews.review.map(clean_text)
before = len(reviews)
reviews = reviews.drop_duplicates(subset="clean").reset_index(drop=True)
print(f"Dropped {before - len(reviews)} duplicate reviews -> {len(reviews)} remain")
reviews["n_words"] = reviews.clean.str.split().str.len()
print("Words per review: median %d, 95th pct %d" % (reviews.n_words.median(), reviews.n_words.quantile(.95)))

## 2.3 TF-IDF and 2.4 classifiers
TF-IDF turns each review into numbers based on its words and two-word phrases. The vectorizer and classifier are combined in one pipeline so LIME can pass in raw text. We train Logistic Regression and Naive Bayes.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

txt_train, txt_test, s_train, s_test = train_test_split(
    reviews.clean, reviews.label, test_size=0.2, stratify=reviews.label, random_state=SEED)

def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True, max_features=100_000)

models = {"Logistic Regression": make_pipeline(tfidf(), LogisticRegression(C=4, solver="liblinear")),
          "Multinomial NB": make_pipeline(tfidf(), MultinomialNB(alpha=0.3))}
for name, m in models.items():
    t = time.time(); m.fit(txt_train, s_train)
    print(f"{name:20s} trained in {time.time() - t:5.1f}s")
for name, m in models.items():
    print(f"{name:20s} train acc {m.score(txt_train, s_train):.4f} | test acc {m.score(txt_test, s_test):.4f}")

### 2.4b Check for overfitting
Logistic Regression scores much higher on training reviews than on test reviews. We pick the regularisation strength `C` with 3-fold cross-validation on the training data, using the same rule as Q1. Smaller `C` means a simpler model. If it is too small the model underfits and both scores drop.

In [ ]:
lr_search = GridSearchCV(
    make_pipeline(tfidf(), LogisticRegression(solver="liblinear")),
    {"logisticregression__C": [0.25, 0.5, 1, 2, 4, 8]},
    cv=3, return_train_score=True, refit=False, n_jobs=3).fit(txt_train, s_train)
lr_cv, lr_choice = pick_simplest(lr_search)
best_C = lr_choice.params["logisticregression__C"]
print("Chosen C:", best_C)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
Cs = lr_cv.param_logisticregression__C.astype(float)
ax.semilogx(Cs, lr_cv.mean_train_score, "o-", label="train")
ax.semilogx(Cs, lr_cv.mean_test_score, "s-", label="cross-validation")
ax.axvline(best_C, ls="--", color="k", lw=0.8, label=f"chosen C={best_C}")
ax.set_xlabel("C (larger = less regularisation)"); ax.set_ylabel("accuracy"); ax.legend()
ax.set_title("Logistic Regression fit check")
plt.tight_layout(); plt.show()

lr_base = models["Logistic Regression"]
pipe = make_pipeline(tfidf(), LogisticRegression(C=best_C, solver="liblinear")).fit(txt_train, s_train)
models["Logistic Regression"] = pipe
display(pd.DataFrame({
    "train acc": [lr_base.score(txt_train, s_train), pipe.score(txt_train, s_train)],
    "CV acc": [lr_cv.loc[Cs == 4, "mean_test_score"].iloc[0], lr_choice.mean_test_score],
    "test acc": [lr_base.score(txt_test, s_test), pipe.score(txt_test, s_test)]},
    index=["first LR (C=4)", f"tuned LR (C={best_C})"]).assign(**{"train-test gap": lambda d: d["train acc"] - d["test acc"]}).round(4))

## 2.5 Evaluate
Accuracy and the classification report on the test set, using the tuned Logistic Regression.

In [ ]:
sent_names = ["negative", "positive"]
for name, m in models.items():
    pred = m.predict(txt_test)
    print(f"=== {name}: accuracy {accuracy_score(s_test, pred):.4f}")
    print(classification_report(s_test, pred, target_names=sent_names, digits=4))

## 2.6 Explain one review with LIME
LIME removes random words from the review and watches how the predicted probability changes. We pick a short, correctly classified review that the model isn't extremely sure about.

In [ ]:
from lime.lime_text import LimeTextExplainer

p_test = pipe.predict_proba(txt_test)[:, 1]
cand = pd.DataFrame({"text": txt_test.values, "y": s_test.values, "p": p_test})
cand["n"] = cand.text.str.split().str.len()
cand["conf"] = np.maximum(cand.p, 1 - cand.p)
cand = cand[(cand.n.between(60, 160)) & (cand.conf.between(0.75, 0.92)) & ((cand.p > .5) == cand.y)]
review = cand.text.iloc[0]
true_lbl = int(cand.y.iloc[0])
p_orig = pipe.predict_proba([review])[0]
pred_lbl = int(p_orig.argmax())
print(f"Actual: {sent_names[true_lbl]} | predicted: {sent_names[pred_lbl]} | P = {p_orig.round(3)}\n")
print(review)

text_explainer = LimeTextExplainer(class_names=sent_names, random_state=SEED)
exp_txt = text_explainer.explain_instance(review, pipe.predict_proba, labels=(1,), num_features=12, num_samples=3000)
print(f"\nLIME local fit R² = {exp_txt.score:.3f}")

### Plot 1 — word importance
Green words push towards positive. Red words push towards negative.

In [ ]:
words = pd.DataFrame(exp_txt.as_list(label=1), columns=["word", "weight"]).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(words.word, words.weight, color=[POS if w > 0 else NEG for w in words.weight])
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("LIME weight  (→ positive   ← negative)"); ax.set_title("LIME word importance")
plt.tight_layout(); plt.show()

### Plot 2 — highlighted text
The same weights shown on the review itself. Darker colour means a stronger effect.

In [ ]:
def highlight(text, weights):
    wmax = max(abs(w) for w in weights.values())
    out = []
    for tok in text.split():
        w = weights.get(tok)
        if w is None:
            out.append(tok)
        else:
            rgb = "46,125,50" if w > 0 else "198,40,40"
            out.append(f'<span style="background:rgba({rgb},{0.15 + 0.75 * abs(w) / wmax:.2f});'
                       f'padding:1px 3px;border-radius:3px" title="{w:+.3f}">{tok}</span>')
    return ('<div style="line-height:2;font-family:sans-serif;font-size:14px;max-width:900px">'
            '<b style="color:#2e7d32">green = positive</b> · <b style="color:#c62828">red = negative</b><br>'
            + " ".join(out) + "</div>")

word_w = dict(exp_txt.as_list(label=1))
display(HTML(highlight(review, word_w)))

## 2.7 Change the two most important words
Remove the two words that most support the prediction, then try replacing them with the model's most opposite word. Check the new predictions.

In [ ]:
sign = 1 if pred_lbl == 1 else -1                     # weights are w.r.t. "positive"
support = sorted(word_w.items(), key=lambda kv: -sign * kv[1])[:2]
support = [(str(w), round(float(v), 4)) for w, v in support]
top2 = [w for w, _ in support]
print("Two most influential words supporting the prediction:", support)

vocab = pipe[0].get_feature_names_out()
coefs = pipe[-1].coef_[0]
unigram = np.array([" " not in v for v in vocab])
opposite = str(vocab[unigram][np.argmin(sign * coefs[unigram])])
print("Strongest opposite-sentiment word in the model:", repr(opposite))

def sub_words(text, targets, repl):
    for t in targets:
        text = re.sub(rf"\b{re.escape(t)}\b", repl, text)
    return re.sub(r"\s+", " ", text).strip()

variants = {
    "original": review,
    f"removed {top2}": sub_words(review, top2, ""),
    f"replaced with '{opposite}'": sub_words(review, top2, opposite),
}
probs = pd.DataFrame({k: pipe.predict_proba([v])[0] for k, v in variants.items()}, index=sent_names).T
probs["predicted"] = probs.idxmax(axis=1)
display(probs.round(4))

### Plot 3 — probability comparison
Predicted probabilities for the original and edited reviews, and the original review under both models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [3, 2]})
probs[sent_names].plot.barh(ax=axes[0], color=[NEG, POS], width=0.7)
axes[0].axvline(0.5, ls="--", color="k", lw=0.8)
axes[0].set_xlim(0, 1); axes[0].set_xlabel("probability"); axes[0].set_title("Logistic Regression — original vs edited review")
axes[0].invert_yaxis()

cmp = pd.DataFrame({n: m.predict_proba([review])[0] for n, m in models.items()}, index=sent_names).T
cmp.plot.bar(ax=axes[1], color=[NEG, POS], rot=0)
axes[1].axhline(0.5, ls="--", color="k", lw=0.8)
axes[1].set_ylim(0, 1); axes[1].set_title("Original review — LR vs Naive Bayes")
plt.tight_layout(); plt.show()

## ★ 2.8 The model's overall word weights
Logistic Regression's coefficients show which words matter across all reviews, not just this one.

In [ ]:
coef_s = pd.Series(coefs, index=vocab).sort_values()
glob = pd.concat([coef_s.head(15), coef_s.tail(15)])
fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(glob.index, glob.values, color=[POS if v > 0 else NEG for v in glob.values])
ax.axvline(0, color="k", lw=0.8); ax.set_title("Logistic Regression: 15 most negative / positive n-grams (global)")
plt.tight_layout(); plt.show()

## ★ 2.9 Check LIME against single-word removal
Remove each word on its own and measure the exact change. If LIME is right, the two rankings should agree.

In [ ]:
from scipy.stats import spearmanr

base = p_orig[1]
loo = {w: base - pipe.predict_proba([sub_words(review, [w], "")])[0, 1] for w in word_w}
chk = pd.DataFrame({"LIME weight": word_w, "leave-one-out ΔP(pos)": loo}).sort_values("LIME weight")
rho = spearmanr(chk["LIME weight"], chk["leave-one-out ΔP(pos)"]).correlation
display(chk.round(4))
print(f"Spearman correlation LIME vs leave-one-out: {rho:.3f}")

## ★ 2.10 Explaining a mistake
Explain the review the model got most confidently wrong.

In [ ]:
wrong = pd.DataFrame({"text": txt_test.values, "y": s_test.values, "p": p_test})
wrong["n"] = wrong.text.str.split().str.len()
wrong = wrong[((wrong.p > .5) != wrong.y) & (wrong.n < 120)]
wrong["err"] = (wrong.p - wrong.y).abs()
bad = wrong.sort_values("err", ascending=False).iloc[0]
print(f"Actual: {sent_names[bad.y]} | predicted P(positive) = {bad.p:.3f}")
exp_bad = text_explainer.explain_instance(bad.text, pipe.predict_proba, labels=(1,), num_features=10, num_samples=3000)
display(HTML(highlight(bad.text, dict(exp_bad.as_list(label=1)))))

Bag-of-words models get fooled when positive words are used in a negative way, for example praising other films.

---
# Question 3 — LIME for Image Data

## 3.1 Load MobileNetV2
A small CNN pretrained on ImageNet's 1,000 classes. It is fast, which matters because LIME calls it about a thousand times. It is not trained here, so overfitting doesn't apply.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

tf.random.set_seed(SEED)
mnet = MobileNetV2(weights="imagenet")
print(f"MobileNetV2: {mnet.count_params():,} parameters, input {mnet.input_shape}")

## 3.2 Select and 3.3 pre-process the image
Download a dog photo, resize it to 224×224 and scale it to 0–1. The classify function converts it to the −1 to 1 range MobileNetV2 expects.

In [ ]:
from tensorflow.keras.utils import get_file, load_img, img_to_array

IMG_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg"
img_path = get_file("labrador.jpg", IMG_URL, cache_dir=".", cache_subdir="data")
img = img_to_array(load_img(img_path, target_size=(224, 224))) / 255.0      # float in [0, 1]

def classify(batch01):
    # batch of [0,1] images -> softmax probabilities over 1000 ImageNet classes
    return mnet.predict(preprocess_input(np.asarray(batch01) * 255.0), verbose=0)

print("image array:", img.shape, img.dtype, f"range [{img.min():.2f}, {img.max():.2f}]")

## 3.4 Top-3 predictions
The three most likely classes and their probabilities.

In [ ]:
probs_img = classify(img[None])[0]
top3 = decode_predictions(probs_img[None], top=3)[0]
top3_ids = list(np.argsort(probs_img)[::-1][:3])
for rank, (wnid, name, p) in enumerate(top3, 1):
    print(f"{rank}. {name:25s} {p:.4f}")
pred_id = top3_ids[0]
pred_name = top3[0][1]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.3]})
axes[0].imshow(img); axes[0].axis("off"); axes[0].set_title("Original input image")
axes[1].barh([t[1] for t in top3][::-1], [t[2] for t in top3][::-1], color="#1565c0")
axes[1].set_xlim(0, 1); axes[1].set_title("Top-3 predicted classes")
plt.tight_layout(); plt.show()

## 3.5 LIME and 3.6 superpixels
The image is split into superpixels (patches of similar colour). LIME hides random patches and learns which ones matter for each class.

In [ ]:
from lime.lime_image import LimeImageExplainer
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import mark_boundaries

quickshift = SegmentationAlgorithm("quickshift", kernel_size=4, max_dist=200, ratio=0.2, random_seed=SEED)
segments = quickshift(img)
print("Number of superpixels:", segments.max() + 1)

img_explainer = LimeImageExplainer(random_state=SEED)
t = time.time()
# labels reversed: LIME keeps only the R² of the last label fitted, so fit the predicted class last
exp_img = img_explainer.explain_instance(img, classify, labels=top3_ids[::-1], top_labels=None, hide_color=0,
                                         num_samples=1000, batch_size=50, segmentation_fn=quickshift,
                                         random_seed=SEED)
print(f"LIME image explanation took {time.time() - t:.1f}s | local R² for '{pred_name}': {exp_img.score:.3f}")

## 3.7 Plots
The original image, the superpixel borders, the patches that support the prediction, and the patches that oppose it.

In [ ]:
def regions(label, positive_only, negative_only=False, k=5, hide_rest=False):
    tmp, mask = exp_img.get_image_and_mask(label, positive_only=positive_only, negative_only=negative_only,
                                           num_features=k, hide_rest=hide_rest, min_weight=0.0)
    return tmp, mask

w_pred = dict(exp_img.local_exp[pred_id])
n_neg = sum(w < 0 for w in w_pred.values())

fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
axes[0].imshow(img); axes[0].set_title("1. Original image")
axes[1].imshow(mark_boundaries(img, segments, color=(1, 1, 0))); axes[1].set_title(f"2. Quickshift superpixels ({segments.max() + 1})")

tmp, mask = regions(pred_id, positive_only=True, k=5, hide_rest=False)
shown = img * 0.3 + 0.7 * img * mask[..., None]          # dim everything outside the mask
axes[2].imshow(mark_boundaries(shown, mask, color=(0, 1, 0)))
axes[2].set_title(f"3. Regions SUPPORTING '{pred_name}'")

if n_neg:
    tmp, mask_n = regions(pred_id, positive_only=False, negative_only=True, k=5)
    over = img.copy(); over[mask_n == 1] = over[mask_n == 1] * 0.4 + np.array([0.8, 0, 0]) * 0.6
    axes[3].imshow(mark_boundaries(over, mask_n, color=(1, 0, 0)))
    axes[3].set_title(f"4. Regions OPPOSING '{pred_name}' ({n_neg} negative)")
else:
    axes[3].imshow(img); axes[3].set_title("4. No opposing regions found")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

### ★ Weight heatmap
Every superpixel coloured by its weight for each of the top-3 classes.

In [ ]:
def weight_map(label):
    m = np.zeros(segments.shape)
    for seg, w in exp_img.local_exp[label]:
        m[segments == seg] = w
    return m

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, lid, (_, name, p) in zip(axes, top3_ids, top3):
    wm = weight_map(lid); lim = np.abs(wm).max()
    ax.imshow(img); im = ax.imshow(wm, cmap="RdYlGn", vmin=-lim, vmax=lim, alpha=0.6)
    ax.set_title(f"{name} (p={p:.2f})"); ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="LIME weight")
plt.suptitle("★ Superpixel weights for each of the top-3 classes")
plt.show()

## 3.8 Mask important regions
Black out the most important patches and re-predict. For comparison, also mask random patches and the opposing patches.

In [ ]:
pos_segs = [s for s, w in exp_img.local_exp[pred_id] if w > 0]     # sorted by |weight| -> keep order
neg_segs = [s for s, w in exp_img.local_exp[pred_id] if w < 0]

def masked(segs):
    out = img.copy(); out[np.isin(segments, segs)] = 0
    return out

tests = {"original": img,
         "mask top-3 supporting": masked(pos_segs[:3]),
         "mask top-5 supporting": masked(pos_segs[:5]),
         "mask 5 random": masked(np.random.RandomState(SEED).choice(segments.max() + 1, 5, replace=False)),
         "mask opposing": masked(neg_segs) if neg_segs else img}
P = classify(np.stack(list(tests.values())))
res = pd.DataFrame({"P(" + pred_name + ")": P[:, pred_id],
                    "new top-1": [decode_predictions(p[None], top=1)[0][0][1] for p in P]}, index=tests.keys())
display(res.round(4))

fig, axes = plt.subplots(1, len(tests), figsize=(18, 3.8))
for ax, (name, im), p in zip(axes, tests.items(), P[:, pred_id]):
    ax.imshow(im); ax.axis("off"); ax.set_title(f"{name}\nP={p:.3f}", fontsize=9)
plt.tight_layout(); plt.show()

## ★ 3.9 Deletion curve
Remove patches one by one in LIME's order and in random order. A faster drop means a better explanation.

In [ ]:
K = min(20, len(pos_segs))
lime_order = [s for s, w in exp_img.local_exp[pred_id]]
lime_curve = classify(np.stack([masked(lime_order[:k]) for k in range(K + 1)]))[:, pred_id]
rand_curves = []
rs = np.random.RandomState(SEED)
for _ in range(5):
    order_r = rs.permutation(segments.max() + 1)
    rand_curves.append(classify(np.stack([masked(order_r[:k]) for k in range(K + 1)]))[:, pred_id])
rand_curve = np.mean(rand_curves, axis=0)

auc = lambda c: np.trapezoid(c, dx=1 / K)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(range(K + 1), lime_curve, "o-", color=NEG, label=f"LIME order (AUC={auc(lime_curve):.3f})")
ax.plot(range(K + 1), rand_curve, "s--", color="gray", label=f"random order, mean of 5 (AUC={auc(rand_curve):.3f})")
ax.set_xlabel("superpixels blacked out"); ax.set_ylabel(f"P({pred_name})"); ax.legend()
ax.set_title("Deletion curve — image explanation")
plt.tight_layout(); plt.show()

## ★ 3.10 Segmentation comparison
Re-run LIME with three different ways of splitting the image into patches and compare the results.

In [ ]:
segmenters = {
    "Quickshift": quickshift,
    "SLIC (n=60)": SegmentationAlgorithm("slic", n_segments=60, compactness=10, sigma=1, start_label=0),
    "Felzenszwalb": SegmentationAlgorithm("felzenszwalb", scale=100, sigma=0.8, min_size=150),
}
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
seg_rows = []
for col, (name, fn) in enumerate(segmenters.items()):
    segs = fn(img)
    t = time.time()
    e = LimeImageExplainer(random_state=SEED).explain_instance(
        img, classify, labels=[pred_id], top_labels=None, hide_color=0, num_samples=500,
        batch_size=50, segmentation_fn=fn, random_seed=SEED)
    _, m = e.get_image_and_mask(pred_id, positive_only=True, num_features=5, hide_rest=False)
    seg_rows.append({"segmenter": name, "superpixels": segs.max() + 1, "local R²": e.score,
                     "seconds": time.time() - t})
    axes[0, col].imshow(mark_boundaries(img, segs, color=(1, 1, 0))); axes[0, col].set_title(f"{name}: {segs.max() + 1} segments")
    axes[1, col].imshow(mark_boundaries(img * 0.3 + 0.7 * img * m[..., None], m, color=(0, 1, 0)))
    axes[1, col].set_title(f"top-5 supporting for '{pred_name}'")
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()
display(pd.DataFrame(seg_rows).round(3))

---
# Summary

- The Random Forest and Logistic Regression were checked for overfitting and tuned with cross-validation on the training data.
- LIME showed sensible reasons for each prediction: cell size and shape, strong sentiment words, the dog's face and chest.
- Removing what LIME marked as important changed the predictions much more than removing random parts.
- LIME is random and depends on choices like sample count and segmentation, so always sanity-check its results.